# Language Model from Scratch: Markov Chains with Word, Character and BPE Tokenizers

**Course project.** We build simple language models from scratch (no neural networks): *n*-gram **Markov chains** that predict the next token from the previous *n* tokens. We train them on Wikipedia text (WikiText-2) and compare

* three **tokenization methods**: word, character, byte-pair encoding (BPE)
* different **context sizes** (Markov order *n* = 1 ... 8)

Every model can complete a sentence from a prompt such as *"mary had a little"*. Generation stops when the model emits an **end-of-sequence (EOS) token**, or at a hard token cap so it can never loop forever.

**Pipeline (the five steps from the brief):**
1. Process the dataset into sentences  2. Split into train / test  3. Create tokenizers  4. Train models for each tokenizer and context size  5. Compare against ground truth and show generated examples.

**How to run:** *Runtime -> Run all*. Everything is controlled from the single configuration cell below. Expect roughly 5-15 minutes on a free Colab CPU (the character model is the slowest).

## 0. Setup

In [ ]:
%pip install -q datasets

import os, re, math, random, time, unicodedata
from collections import Counter, defaultdict

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # silence the unauthenticated-request warning (public dataset)

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

### Configuration
All tunable choices live here. The reasons for each are explained in section 8.

In [ ]:
SEED         = 42
MAX_WORDS    = 40            # sentence length filter (words)
MIN_WORDS    = 4
TEST_FRAC    = 0.10          # 90 / 10 sentence-level split

N_TRAIN      = 20000         # training sentences used by every model (same subset -> fair comparison)
WORD_MIN_FREQ = 2            # words seen fewer times in training become <UNK>
BPE_MERGES   = 1000          # number of BPE merge operations
MAX_ORDER    = {"word": 4, "bpe": 5, "char": 8}    # highest context size trained per tokenizer

N_LM_EVAL    = 2000          # test sentences used for bits-per-character
N_COMPLETION = 300           # test sentences used for the completion test
GEN_CAP      = {"word": 50, "bpe": 100, "char": 300}   # hard cap on generated tokens

## 1. Process the dataset into sentences

We use **WikiText-2 (raw)** from Hugging Face: real Wikipedia text, small enough to train quickly.

Cleaning rules (the brief asks for alphanumeric characters, white-space and punctuation only):
* drop blank lines and `= Heading =` lines
* strip accents (`senjō` -> `senjo`) and drop other non-ASCII symbols (e.g. Japanese text)
* lower-case everything (a smaller vocabulary matters a lot for Markov models, which get sparse quickly)
* undo WikiText's tokenisation artefacts: `role @-@ playing` -> `role-playing`, `1 @,@ 000` -> `1,000`, `did n't` -> `didn't`, `game 's` -> `game's`
* keep only `a-z 0-9`, white-space and the punctuation `. , ! ? ; : ' -`
* split into sentences after `.`, `!` or `?`, and keep sentences of 4-40 words

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
# Merge the pre-made splits into one pool: we do our own sentence-level split in the next cell.
lines = [r["text"] for split in ("train", "validation", "test") for r in ds[split]]
print("raw lines:", len(lines))

def clean_line(line):
    line = line.strip()
    if not line or line.startswith("="):                 # blank lines and headings
        return ""
    line = unicodedata.normalize("NFKD", line).encode("ascii", "ignore").decode()   # senjo -> senjo, drops CJK
    line = line.lower()
    line = re.sub(r"\s*@([-,.])@\s*", r"\1", line)        # "role @-@ playing" -> "role-playing"
    line = re.sub(r"[^a-z0-9\s.,!?;:'\-]", " ", line)     # alphanumeric + white-space + punctuation only
    line = re.sub(r"\s+(n't|'(?:s|re|ve|ll|d|m))\b", r"\1", line)   # "did n't" -> "didn't"
    line = re.sub(r"\s+", " ", line).strip()
    line = re.sub(r"\s+([.,!?;:])", r"\1", line)          # "word ." -> "word."
    return line

def split_sentences(text):
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]

sentences = []
for line in lines:
    c = clean_line(line)
    if c:
        sentences += split_sentences(c)
sentences = [s for s in sentences if MIN_WORDS <= len(s.split()) <= MAX_WORDS]

print("sentences:", len(sentences))
for s in sentences[:4]:
    print(" -", s)

## 2. Split into train and test

A random **sentence-level** split (90 % train / 10 % test) with a fixed seed, so results are reproducible and no test sentence appears in training.

In [ ]:
random.seed(SEED)
random.shuffle(sentences)
cut = int((1 - TEST_FRAC) * len(sentences))
train_sents, test_sents = sentences[:cut], sentences[cut:]
train_sub = train_sents[:N_TRAIN]          # the subset every model is trained on
print(f"train: {len(train_sents)}   test: {len(test_sents)}   used for training the models: {len(train_sub)}")

## 3. Tokenizers

All three tokenizers share one interface:
* `encode(sentence)` -> list of integer ids, **always ending with the `<EOS>` id**
* `decode(ids)` -> text, stopping at `<EOS>`

| Tokenizer | Token | Notes |
|---|---|---|
| **Character** | one character | tiny vocabulary, very long sequences |
| **Word** | a word or a punctuation mark | huge vocabulary; words rarer than `WORD_MIN_FREQ` become `<UNK>` |
| **BPE** | learned sub-word piece | starts from characters and repeatedly merges the most frequent adjacent pair; merges never cross word boundaries and the leading space is part of the token, so decoding is exact |

In [ ]:
EOS, UNK = "<EOS>", "<UNK>"

class Tokenizer:
    def build_vocab(self, tokens):
        self.itos = [EOS, UNK] + sorted(set(tokens) - {EOS, UNK})
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.eos_id, self.unk_id = self.stoi[EOS], self.stoi[UNK]

    def encode(self, sentence):
        return [self.stoi.get(t, self.unk_id) for t in self.tokenize(sentence)] + [self.eos_id]

    def decode(self, ids):
        toks = []
        for i in ids:
            if i == self.eos_id:
                break
            toks.append(self.itos[i])
        return self.detokenize(toks)

    # Hooks used when a prompt ends on a complete word and generation must start a NEW word.
    def boundary_ids(self):          # ids appended after such a prompt
        return []
    def starts_new_word(self, t):    # may token id t begin a new word?
        return True


class CharTokenizer(Tokenizer):
    def fit(self, sentences):
        self.build_vocab(c for s in sentences for c in s)
        return self
    def boundary_ids(self):  return [self.stoi[" "]] if " " in self.stoi else []
    def tokenize(self, s):    return list(s)
    def detokenize(self, t):  return "".join(t)


class WordTokenizer(Tokenizer):
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
    def tokenize(self, s):
        return re.findall(r"\w+(?:['.,]\w+)*|[^\w\s]", s)          # keeps didn't, 1,000, 3.5 whole
    def fit(self, sentences):
        counts = Counter(w for s in sentences for w in self.tokenize(s))
        self.build_vocab(w for w, c in counts.items() if c >= self.min_freq)
        return self
    def detokenize(self, t):
        text = " ".join(t)
        text = re.sub(r"\s+([.,!?;:])", r"\1", text)
        return re.sub(r"\s*-\s*", "-", text)

In [ ]:
class BPETokenizer(Tokenizer):
    def __init__(self, num_merges=1000):
        self.num_merges = num_merges

    def pretokenize(self, s):
        return re.findall(r" ?\w+| ?[^\w\s]", s)               # each piece keeps its leading space

    def fit(self, sentences):
        word_freq = Counter(w for s in sentences for w in self.pretokenize(s))
        words = {w: list(w) for w in word_freq}                # word -> current list of symbols
        self.merges = []
        for _ in range(self.num_merges):
            pairs = Counter()
            for w, syms in words.items():
                for a, b in zip(syms, syms[1:]):
                    pairs[(a, b)] += word_freq[w]
            if not pairs:
                break
            best = max(pairs, key=pairs.get)                   # most frequent adjacent pair
            self.merges.append(best)
            for w, syms in words.items():                      # apply the merge everywhere
                i, out = 0, []
                while i < len(syms):
                    if i < len(syms) - 1 and (syms[i], syms[i + 1]) == best:
                        out.append(syms[i] + syms[i + 1]); i += 2
                    else:
                        out.append(syms[i]); i += 1
                words[w] = out
        self.rank = {p: r for r, p in enumerate(self.merges)}
        self._cache = {}
        self.build_vocab(sym for syms in words.values() for sym in syms)
        return self

    def _bpe(self, word):
        if word in self._cache:
            return self._cache[word]
        syms = list(word)
        while len(syms) > 1:
            cand = [(self.rank[p], i) for i, p in enumerate(zip(syms, syms[1:])) if p in self.rank]
            if not cand:
                break
            _, i = min(cand)                                   # apply the earliest-learned merge first
            syms[i:i + 2] = [syms[i] + syms[i + 1]]
        self._cache[word] = syms
        return syms

    def tokenize(self, s):    return [t for w in self.pretokenize(s) for t in self._bpe(w)]
    def detokenize(self, t):  return "".join(t).strip()
    def starts_new_word(self, t):                              # pieces carry their leading space
        s = self.itos[t]
        return t == self.eos_id or s[0] == " " or not s[0].isalnum()

In [ ]:
t0 = time.time()
tokenizers = {
    "word": WordTokenizer(WORD_MIN_FREQ).fit(train_sub),
    "bpe":  BPETokenizer(BPE_MERGES).fit(train_sub),
    "char": CharTokenizer().fit(train_sub),
}
print(f"tokenizers fitted in {time.time() - t0:.0f}s (on the same {len(train_sub)} training sentences as the models)\n")

example = test_sents[0]
print("Example sentence:", example)
rows = []
check = test_sents[:1000]
for name, tok in tokenizers.items():
    ids = tok.encode(example)
    toks = [tok.itos[i] for i in ids]
    rt = sum(tok.decode(tok.encode(s)) == s for s in check) / len(check)
    avg_len = sum(len(tok.encode(s)) for s in check) / len(check)
    rows.append(dict(tokenizer=name, vocab_size=len(tok.itos), avg_tokens_per_sentence=round(avg_len, 1),
                     exact_roundtrip=f"{rt:.1%}", example_tokens=toks))
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows).set_index("tokenizer")

*Reading the table:* character and BPE decode **exactly** (100 % round-trip). The word tokenizer is below 100 % **by design**: any sentence containing a rare word comes back with `<UNK>` in its place. Vocabulary size and tokens-per-sentence show the trade-off: characters give a tiny vocabulary and long sequences, words the reverse, BPE sits in between.

## 4. The Markov-chain language model

A Markov chain of **order n** predicts the next token from the previous *n* tokens:

`P(next token | last n tokens)` is estimated by counting, in the training data, which tokens followed each context.

Implementation choices:
* **One model per tokenizer, with the order chosen at use time.** The model stores counts for every context length 0 ... `max_order`, so a single training pass supports all orders up to the maximum.
* **Sentence start padding.** Each sentence is padded with `BOS` markers so even the first token has a context of full length *n*.
* **Generation** repeatedly samples the next token from the counts, appends it, and feeds the result back in, until **`<EOS>`** is sampled or the **hard cap** is reached. `<UNK>` is never sampled. Prompts end on a complete word, so the first generated token must start a *new* word (otherwise character and BPE models would glue letters onto the prompt's last word).
* **Back-off for unseen contexts.** If the last *n* tokens never occurred in training, the model falls back to the longest shorter context that did (down to a unigram). Without this a Markov chain would get stuck.
* **Smoothing for evaluation.** To score held-out text we need non-zero probabilities, so the probability of each token is **Witten-Bell interpolated** across orders (each order is blended with the next-lower one; the lowest level is an add-one unigram).

In [ ]:
BOS = -1   # padding id for "before the start of the sentence"

class MarkovLM:
    def __init__(self, max_order):
        self.max_order = max_order
        # counts[k][context_tuple][next_id] = count, where len(context_tuple) == k
        self.counts = [defaultdict(Counter) for _ in range(max_order + 1)]
        self._tc = {}                                   # cache of context totals

    def fit(self, encoded_sents):
        N = self.max_order
        for ids in encoded_sents:
            seq = [BOS] * N + ids
            for i in range(N, len(seq)):
                tok = seq[i]
                for k in range(N + 1):
                    self.counts[k][tuple(seq[i - k:i])][tok] += 1
        return self

    def _ctx(self, ids, k):
        return tuple(([BOS] * k + ids)[-k:]) if k else ()

    def next_items(self, tok, ids, order, new_word=False):
        """(token, count) pairs from the longest seen context of length <= order (back-off).
        <UNK> is never offered; with new_word=True only tokens that can start a new word are."""
        for k in range(min(order, self.max_order), -1, -1):
            c = self.counts[k].get(self._ctx(ids, k))
            if not c:
                continue                                  # unseen context -> back off to a shorter one
            items = [(t, n) for t, n in c.items()
                     if t != tok.unk_id and (not new_word or tok.starts_new_word(t))]
            if items:
                return items
        return []

    def complete(self, tok, prompt, order, max_tokens=60, seed=None):
        """Continue `prompt` (which ends on a complete word). Returns (full_text, continuation_only)."""
        rng = random.Random(seed)
        ids = tok.encode(prompt)[:-1] + tok.boundary_ids()    # drop the EOS that encode() appends
        start = len(ids)
        for step in range(max_tokens):                    # hard cap: generation can never loop forever
            items = self.next_items(tok, ids, order, new_word=(step == 0))
            if not items:
                break
            nxt = rng.choices([t for t, _ in items], weights=[n for _, n in items])[0]
            if nxt == tok.eos_id:                         # the model decided the sentence is over
                break
            ids.append(nxt)
        return tok.decode(ids), tok.decode(ids[start:])

    def generate(self, tok, prompt, order, max_tokens=60, seed=None):
        return self.complete(tok, prompt, order, max_tokens, seed)[0]

    def _total(self, k, ctx, c):
        key = (k, ctx)
        if key not in self._tc:
            self._tc[key] = sum(c.values())
        return self._tc[key]

    def prob_all(self, prefix, tok_id, V):
        """Witten-Bell interpolated P(tok_id | prefix) for every order 0..max_order, in one pass."""
        N = self.max_order
        padded = [BOS] * N + prefix
        c0 = self.counts[0][()]
        p = (c0.get(tok_id, 0) + 1) / (self._total(0, (), c0) + V)     # add-one unigram
        out = [p]
        for k in range(1, N + 1):
            ctx = tuple(padded[-k:])
            c = self.counts[k].get(ctx)
            if c:
                T = len(c)                                              # number of distinct continuations
                p = (c.get(tok_id, 0) + T * p) / (self._total(k, ctx, c) + T)
            out.append(p)
        return out

In [ ]:
models = {}
for name, tok in tokenizers.items():
    t0 = time.time()
    encoded = [tok.encode(s) for s in train_sub]
    models[name] = MarkovLM(MAX_ORDER[name]).fit(encoded)
    print(f"{name:5} max_order={MAX_ORDER[name]}  trained on {len(encoded)} sentences in {time.time() - t0:.0f}s")

## 5. Sentence completion demo

Each model is given a prompt and continues it until `<EOS>`. Low orders give gibberish; higher orders become fluent but increasingly copy the training text (quantified in section 6).

In [ ]:
prompts = ["mary had a little", "he was born in"]
for prompt in prompts:
    print(f"\n######## prompt: '{prompt}' ########")
    for name, tok in tokenizers.items():
        print(f"\n=== {name} tokenizer ===")
        for order in range(1, MAX_ORDER[name] + 1):
            out = models[name].generate(tok, prompt, order, max_tokens=GEN_CAP[name], seed=1)
            print(f"n={order}: {out}")

## 6. Evaluation against the test set (ground truth)

Three metrics, all computed on **held-out test sentences** the models never saw:

1. **Bits per character (bpc)** - how well the model predicts the real text (lower is better). Raw perplexity is not comparable across tokenizers because their tokens have different sizes, so we divide the total code length by the number of *characters* (`<EOS>` counts as one character). *Caveat:* the word model absorbs rare words into `<UNK>`, which flatters its bpc.
2. **Sentence completion** - the model gets the **first half** of a test sentence and writes the rest; we compare with the true second half using **next-word accuracy** (is the first generated word correct?) and **word-overlap F1** against the true ending. Markov models cannot plan ahead, so low absolute values are expected - the comparison between settings is what matters.
3. **Copy rate** - the share of generated sentences that are **verbatim copies of a training sentence**, which measures memorisation.

In [ ]:
rng = random.Random(SEED)
lm_sample   = rng.sample(test_sents, min(N_LM_EVAL, len(test_sents)))
comp_sample = rng.sample([s for s in test_sents if len(s.split()) >= 6], N_COMPLETION)
train_text_set = set(train_sub)

def eval_lm(name):
    """Bits per character and token perplexity for every order, in one pass over the test sample."""
    tok, m = tokenizers[name], models[name]
    V, N = len(tok.itos), m.max_order
    bits, ntok, nchar = [0.0] * (N + 1), 0, 0
    for s in lm_sample:
        ids = tok.encode(s)
        nchar += len(s) + 1                                  # +1 for the EOS
        for i, t in enumerate(ids):
            ps = m.prob_all(ids[:i], t, V)
            for k in range(1, N + 1):
                bits[k] -= math.log2(ps[k])
            ntok += 1
    return {k: (bits[k] / nchar, 2 ** (bits[k] / ntok)) for k in range(1, N + 1)}

def eval_completion(name, order):
    tok, m = tokenizers[name], models[name]
    first_hit = f1_sum = copies = 0
    for j, s in enumerate(comp_sample):
        words = s.split()
        k = max(2, len(words) // 2)
        prompt, truth = " ".join(words[:k]), words[k:]
        full, cont = m.complete(tok, prompt, order, GEN_CAP[name], seed=j)
        comp = cont.split()
        first_hit += bool(comp) and comp[0] == truth[0]
        overlap = sum((Counter(comp) & Counter(truth)).values())
        if overlap:
            pr, rc = overlap / len(comp), overlap / len(truth)
            f1_sum += 2 * pr * rc / (pr + rc)
        copies += full in train_text_set
    n = len(comp_sample)
    return first_hit / n, f1_sum / n, copies / n

rows = []
for name in tokenizers:
    t0 = time.time()
    lm_scores = eval_lm(name)
    for order in range(1, MAX_ORDER[name] + 1):
        bpc, ppl = lm_scores[order]
        acc, f1, copy = eval_completion(name, order)
        rows.append(dict(tokenizer=name, order=order, bpc=bpc, token_ppl=ppl,
                         next_word_acc=acc, overlap_f1=f1, copy_rate=copy))
    print(f"{name}: evaluated in {time.time() - t0:.0f}s")

results = pd.DataFrame(rows)
results.round(3)

In [ ]:
best = results.loc[results.groupby("tokenizer")["bpc"].idxmin()].set_index("tokenizer")
print("Best order per tokenizer (lowest bits per character):")
best[["order", "bpc", "next_word_acc", "overlap_f1", "copy_rate"]].round(3)

### Plots
The same numbers as the table above, plotted against the context size *n*. Each series has its own colour **and** marker shape, and is labelled at the end of its line.

In [ ]:
COLORS  = {"word": "#2a78d6", "bpe": "#eb6834", "char": "#1baf7a"}
MARKERS = {"word": "o", "bpe": "s", "char": "^"}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"

def panel(ax, col, title, ylabel, pct=False):
    for name in ["word", "bpe", "char"]:
        d = results[results.tokenizer == name]
        ax.plot(d.order, d[col], color=COLORS[name], marker=MARKERS[name], lw=2, ms=7,
                markeredgecolor=SURFACE, markeredgewidth=1.5, label=name)
        ax.annotate(name, (d.order.iloc[-1], d[col].iloc[-1]), xytext=(8, 0),
                    textcoords="offset points", va="center", fontsize=10, color=INK)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("context size n (tokens)", color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.set_xticks(range(1, int(results.order.max()) + 1))
    ax.margins(x=0.14)
    if pct:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, lw=1)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
fig.patch.set_facecolor(SURFACE)
panel(axes[0], "bpc", "Bits per character on test text (lower = better)", "bits per character")
panel(axes[1], "copy_rate", "Generated sentences copied from training", "share of generated sentences", pct=True)
panel(axes[2], "overlap_f1", "Completion: word overlap with the true ending", "overlap F1")
plt.tight_layout()
plt.show()

## 7. Generated sentences vs ground truth

For a few test sentences: the **prompt** (first half), the **true ending**, and what each tokenizer's model writes at its **best order** (lowest bits per character).

In [ ]:
best_order = best["order"].to_dict()
for j in range(6):
    s = comp_sample[j]
    words = s.split()
    k = max(2, len(words) // 2)
    prompt, truth = " ".join(words[:k]), " ".join(words[k:])
    print(f"PROMPT : {prompt}")
    print(f"TRUTH  : {truth}")
    for name in tokenizers:
        _, cont = models[name].complete(tokenizers[name], prompt, best_order[name], GEN_CAP[name], seed=j)
        print(f"{name:6} (n={best_order[name]}): {cont.strip()}")
    print()

In [ ]:
results.to_csv("results.csv", index=False)   # handy for the report

## 8. Design decisions and why

| Decision | Choice | Why |
|---|---|---|
| **Dataset** | WikiText-2 (raw), all three pre-made splits pooled | Real Wikipedia text, as suggested; small enough for fast experiments. Pooled because we split by sentence ourselves. |
| **Cleaning** | lower-case, ASCII only, keep `a-z 0-9 . , ! ? ; : ' -`, undo `@-@`-style artefacts | Meets the brief (alphanumeric + white-space + punctuation) and keeps the vocabulary small. Accents are stripped rather than deleted so words are not split. |
| **Sentence split** | after `.`, `!`, `?`; 4-40 words | Simple and fast. Known limitation: abbreviations such as `lit.` are treated as sentence ends. |
| **Train/test split** | random, sentence level, 90/10, seed 42 | No leakage of test sentences into training; reproducible. |
| **Training size** | 20,000 sentences for every model | Character models at high orders store roughly one entry per character per order, which exhausts RAM on the full set. The same subset for all tokenizers keeps the comparison fair. |
| **EOS token** | appended to every sentence; generation stops when it is sampled | Required by the brief; it lets the model decide when a sentence ends. |
| **Unknown words** | words seen < 2 times become `<UNK>` (never generated) | Rare words would otherwise explode the vocabulary and make counts too sparse. |
| **BPE** | 1,000 merges, merges within words only, space kept in the token | A middle ground between characters and words; makes decoding exact. |
| **Context sizes** | word 1-4, BPE 1-5, character 1-8 | Smaller tokens need longer contexts to cover the same span of text. |
| **Unseen contexts** | back off to the longest shorter seen context | A plain Markov chain gets stuck on contexts it has never seen. |
| **Loop protection** | hard cap on generated tokens (50 / 100 / 300 for word / BPE / char) | Required by the brief. |
| **Scoring** | Witten-Bell interpolation, bits per *character* | Non-zero probabilities for unseen events; bpc is comparable across tokenizers. |

## 9. Results and discussion

*Fill this section in from the table and plots above.* Suggested points:

* **Context size.** How do bits per character change with *n* for each tokenizer? Where is the minimum, and why does it get worse afterwards (sparse contexts: most long contexts were seen only once or never)?
* **Tokenizers.** Which tokenizer is best at which order? Compare vocabulary size (section 3) with how long a context each needs. Remember the `<UNK>` caveat for the word model.
* **Memorisation.** How does the copy rate grow with *n*? At which order do generated sentences stop being new?
* **Completion quality.** Are overlap and next-word accuracy higher at higher orders, and does this just reflect copying?
* **Qualitative examples.** Quote two or three generated sentences from sections 5 and 7: gibberish at low *n*, plausible but incoherent at medium *n*, fluent copies at high *n*.
* **Limitations.** Markov chains have no long-range memory or planning; small training subset; sentence splitter; evaluation on a sample of the test set; single random seed for generation.